# 📊 INF01090 - Ciência de Dados - Regression Techniques

**House Prices - Advanced Regression Techniques - Kaggle-Style Competition**

House Prices – Advanced Regression Techniques (<https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/>) is a long-running Kaggle competition that challenges participants to predict the sale prices of homes in Ames, Iowa, using advanced regression models. It is one of Kaggle’s most popular “Getting Started” challenges, designed to build practical skills in data cleaning, feature engineering, and predictive modelling.

## Key facts

- **Platform:** Kaggle
- **Launch year:** 2016
- **Original dataset:** 1,460 training and 1,459 test homes
- **Features:** 79 explanatory variables
- **Primary goal:** Predict house sale price


## 📂 Dataset and Objective

The dataset, compiled by Dean De Cock as an update to the classic Boston Housing dataset, describes almost every aspect of residential homes—covering lot size, room counts, materials, neighbourhood, and more. The task is to use these features to predict each property's final sale price, making it a supervised regression problem that blends statistical and machine learning techniques.

The file **`data_description.txt`** has the full description of each column.

## 🛠 Skills and techniques

This competition is widely used to practise:

- **Feature engineering:** handling missing data, encoding categorical variables, transforming skewed features
- **Model building:** experimenting with algorithms such as linear regression, ridge, lasso, random forest, and gradient boosting
- **Evaluation:** models are ranked by **RMSE on log-transformed prices** in our local competition setup


## 🎯 Assignment Goal

Your goal is to build a regression pipeline that predicts **`SalePrice`** for the hidden test set.

This is not only a leaderboard exercise. You should use this assignment to demonstrate that you understand:

- data cleaning for tabular data
- feature encoding and transformation
- regression modeling
- error analysis
- the effect of different design choices on predictive performance


## 📁 Files You Will Receive

You should work only with the files distributed for this lab:

- **`train_student.csv`** — training data with the target column
- **`test_student.csv`** — test data without the target column
- **`submission_template.csv`** — expected format for submission
- **`data_description.txt`** — attribute descriptions

Do **not** use Kaggle's original public test split for submission. The grading app uses a **custom hidden split** created for this class.


## 🏁 Submission and Leaderboard

Submissions are evaluated in the local grading app:

<https://labo5inf01090-huzhzhojtbeqqknm7duabo.streamlit.app/>

The app expects a CSV file with exactly these columns:

```csv
Id,prediction
1461,210000
1462,179500
1463,220000
```

Rules:

- `Id` must match the IDs in **`test_student.csv`**
- `prediction` must contain one numeric prediction per row
- all test rows must be present
- predictions for `SalePrice` should be non-negative


## 📌 What You Must Deliver

Each group must submit:

1. **A prediction file** for the leaderboard  
2. **This notebook** (completed, with code, outputs, and short explanations)  
3. **A short report section in the notebook** explaining:
   - preprocessing choices
   - feature engineering
   - model(s) tested
   - final model used
   - interpretation of the obtained score


## 👥 Group Work

- Work in groups of up to 4
- All members of the group must understand the final solution
- Use a consistent team name in the leaderboard
- The same team may submit multiple times; the leaderboard keeps the best score


## 📊 Grading Criteria

Your grade will not depend only on leaderboard position. The first three places will get additional grade.

Important:
- a top leaderboard score with poor documentation is **not enough**
- a strong notebook with solid methodology can still receive a high grade even if it is not the top-ranked solution


## 🚦 Recommended Workflow

A good workflow for this assignment is:

1. Inspect the training data
2. Identify numeric and categorical variables
3. Handle missing values
4. Encode categorical variables
5. Optionally transform skewed variables
6. Build a baseline regression model
7. Evaluate improvements using validation on the training set
8. Train your final model on the full student training set
9. Predict on `test_student.csv`
10. Submit the predictions to the leaderboard


## ⚠️ Restrictions and Good Practice

- Do not manually inspect or reconstruct the hidden target values
- Do not hard-code predictions
- Do not submit malformed files to probe the scorer
- Do not use the leaderboard as your only validation method

Recommended:
- create your own validation split from `train_student.csv`
- compare models locally before submitting
- submit only meaningful improvements


## 🧪 Suggested Experiments

You may explore ideas such as:

- dropping columns with many missing values
- imputing missing values numerically and categorically
- one-hot encoding categorical variables
- applying `log1p(SalePrice)` during training
- trying different regularization strengths
- comparing linear and non-linear models
- checking whether some features are highly skewed


## 🧭 Starter Checklist

Before your first submission, verify that:

- [ ] `train_student.csv` loads correctly
- [ ] `test_student.csv` has the same predictor columns as expected
- [ ] your preprocessing works for both train and test
- [ ] your model produces one prediction per test row
- [ ] the output file has exactly two columns: `Id`, `prediction`
- [ ] all predictions are numeric
- [ ] all predictions are non-negative


## 🐍 Suggested Notebook Structure

You may organize your work using sections such as:

1. Data loading
2. Exploratory inspection
3. Missing-value handling
4. Feature encoding
5. Train/validation split
6. Baseline model
7. Improved model
8. Final training and test prediction
9. Submission file generation
10. Reflection


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor


## 1. Load the data

Update the paths if necessary.


In [2]:
train_df = pd.read_csv("train_student.csv")
test_df = pd.read_csv("test_student.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


Train shape: (1022, 81)
Test shape: (438, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,136,20,RL,80.0,10400,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,5,2008,WD,Normal,174000
1,1453,180,RM,35.0,3675,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2006,WD,Normal,145000
2,763,60,FV,72.0,8640,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,6,2010,Con,Normal,215200
3,933,20,RL,84.0,11670,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,3,2007,WD,Normal,320000
4,436,60,RL,43.0,10667,Pave,NaN,IR2,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2009,ConLw,Normal,212000


# 2. EDA(Exploratory Data Analysis)

# How many missing values are there(%)?

In [3]:
# How many missing values are there in each column?
missing_percent = (train_df.isnull().sum() / len(train_df)) * 100

missing_data = missing_percent[missing_percent > 0].sort_values(ascending=False)

print("Missing Values (%):")
if not missing_data.empty:
    print(missing_data.map("{:.2f}%".format))
else:
    print("No missing values found!")

Missing Values (%):
PoolQC          99.51%
MiscFeature     96.09%
Alley           93.54%
Fence           80.23%
MasVnrType      57.73%
FireplaceQu     47.65%
LotFrontage     18.59%
GarageType       5.28%
GarageYrBlt      5.28%
GarageFinish     5.28%
GarageQual       5.28%
GarageCond       5.28%
BsmtCond         2.54%
BsmtFinType1     2.54%
BsmtExposure     2.54%
BsmtQual         2.54%
BsmtFinType2     2.54%
MasVnrArea       0.29%
Electrical       0.10%
dtype: object


Excluiding columns with >=45% of missing values and lines with less than 0.5%

In [4]:
train_df.drop(columns=['PoolQC', 'MiscFeature', 'Alley', 'Fence'], inplace=True)
train_df.dropna(subset=['Electrical', 'MasVnrArea'], inplace=True)

# Heatmap (Correlation using spearman)

# Separating Data into numerical and categorical

In [5]:

numeric_cols = train_df.select_dtypes(include=['number']).columns
categorical_cols = train_df.select_dtypes(include=['object']).columns

## 2.1 Handle Missing Numerical Values

In [6]:
for col in numeric_cols:
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna(0)

## 2.2 Handle Missing Categorical Values

In [7]:
for col in train_df[categorical_cols]:
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna('None')

# 3 Feature Encoding

Identify target and predictors

In [8]:
target_col = "SalePrice"
id_col = "Id"

X = train_df.drop(columns=[target_col])
y = train_df[target_col].copy()

print("Target summary:")
display(y.describe())


Target summary:


count      1018.000000
mean     181218.734774
std       77739.066613
min       34900.000000
25%      130000.000000
50%      164995.000000
75%      215000.000000
max      745000.000000
Name: SalePrice, dtype: float64

## 3.1 Encoding Categorical Variables

In [ ]:
QUALITY_MAP = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

ORDINAL_MAPS = {
    "ExterQual":    QUALITY_MAP,
    "ExterCond":    QUALITY_MAP,
    "BsmtQual":     QUALITY_MAP,
    "BsmtCond":     QUALITY_MAP,
    "HeatingQC":    QUALITY_MAP,
    "KitchenQual":  QUALITY_MAP,
    "GarageQual":   QUALITY_MAP,
    "GarageCond":   QUALITY_MAP,
    "FireplaceQu":  QUALITY_MAP,
    "PoolQC":       QUALITY_MAP,
    "BsmtExposure": {"None": 0, "No": 0, "Mn": 1, "Av": 2, "Gd": 3},
    "BsmtFinType1": {"None": 0, "Unf": 0, "LwQ": 1, "Rec": 2, "BLQ": 3, "ALQ": 4, "GLQ": 5},
    "BsmtFinType2": {"None": 0, "Unf": 0, "LwQ": 1, "Rec": 2, "BLQ": 3, "ALQ": 4, "GLQ": 5},
    "GarageFinish": {"None": 0, "Unf": 0, "RFn": 1, "Fin": 2},
    "LotShape":     {"IR3": 0, "IR2": 1, "IR1": 2, "Reg": 3},
    "LandSlope":    {"Sev": 0, "Mod": 1, "Gtl": 2},
    "PavedDrive":   {"N": 0, "P": 1, "Y": 2},
    "Functional":   {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3,
                     "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7},
}  
# Variáveis binárias simples
BINARY_MAPS = {
    "CentralAir": {"Y": 1, "N": 0},
    "Street":     {"Pave": 1, "Grvl": 0},
}

# Nominais para one-hot (excluindo Neighborhood que usa target encoding)
NOMINAL_COLS = [
    "MSZoning", "LotConfig", "LandContour", "Neighborhood",
    "Condition1", "Condition2", "BldgType", "HouseStyle",
    "RoofStyle", "RoofMatl",
    "Exterior1st", "Exterior2nd", "MasVnrType",
    "Foundation", "Heating",
    "GarageType", "MiscFeature",
    "SaleType", "SaleCondition",
    "Electrical", "Alley", "Fence",
]


## 3. Build a local validation split

Use this split to compare models **before** submitting to the leaderboard.


In [10]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_valid.shape)


(814, 76) (204, 76)


## 5. Create a preprocessing pipeline

You may improve this pipeline as part of the assignment.


In [11]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])


NameError: name 'numeric_features' is not defined

## 6. Baseline model

Start with a simple model.


In [ ]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

baseline_model.fit(X_train, y_train)
pred_valid_baseline = baseline_model.predict(X_valid)

rmse_baseline = mean_squared_error(y_valid, pred_valid_baseline) ** 0.5
mae_baseline = mean_absolute_error(y_valid, pred_valid_baseline)
r2_baseline = r2_score(y_valid, pred_valid_baseline)

print("Baseline RMSE:", rmse_baseline)
print("Baseline MAE :", mae_baseline)
print("Baseline R²  :", r2_baseline)


Baseline RMSE: 30680.926083805487
Baseline MAE : 21579.421029016994
Baseline R²  : 0.8488799165654537


: 

: 

: 

: 

: 

: 

: 

: 

## 7. Improved model

Try at least one stronger model and compare the result.


In [ ]:
improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

improved_model.fit(X_train, y_train)
pred_valid_improved = improved_model.predict(X_valid)

rmse_improved = mean_squared_error(y_valid, pred_valid_improved) ** 0.5
mae_improved = mean_absolute_error(y_valid, pred_valid_improved)
r2_improved = r2_score(y_valid, pred_valid_improved)

print("Improved RMSE:", rmse_improved)
print("Improved MAE :", mae_improved)
print("Improved R²  :", r2_improved)


Improved RMSE: 25911.333275388366
Improved MAE : 17194.49053658537
Improved R²  : 0.8922133990726447


: 

: 

: 

: 

: 

: 

: 

: 

## 8. Compare models

Briefly discuss the difference between the baseline and the improved model.


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "Improved"],
    "RMSE": [rmse_baseline, rmse_improved],
    "MAE": [mae_baseline, mae_improved],
    "R2": [r2_baseline, r2_improved],
})

comparison


,Model,RMSE,MAE,R2
0,Baseline,30680.926084,21579.421029,0.848880
1,Improved,25911.333275,17194.490537,0.892213


: 

: 

: 

: 

: 

: 

: 

: 

**Write a short discussion here.**

- Which model performed better?
- Was the improvement large or small?
- What might explain the difference?


## 9. Train the final model on the full student training set

Choose your final model and fit it using all available labeled data.


In [ ]:
final_model = improved_model  # change if needed

final_model.fit(X, y)
test_predictions = final_model.predict(test_df)

submission = pd.DataFrame({
    id_col: test_df[id_col],
    "prediction": np.maximum(test_predictions, 0)  # keep predictions non-negative
})

submission.head()


,Id,prediction
0,893,140478.416667
1,1106,316577.800000
2,414,117722.000000
3,523,152438.516667
4,1037,326923.336667


: 

: 

: 

: 

: 

: 

: 

: 

## 10. Save the submission file


In [ ]:
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")


Saved submission.csv


: 

: 

: 

: 

: 

: 

: 

: 

## 11. Submit to the leaderboard

Upload `submission.csv` to:

<https://labo5inf01090-huzhzhojtbeqqknm7duabo.streamlit.app/>

After submitting, record your score below.


**Leaderboard score(s):**

- First submission:
- Best submission:
- Final submitted model:


## 12. Final reflection

Write a short final reflection addressing:

- what preprocessing choices were most important
- whether feature engineering helped
- what model worked best for your group
- what you would try next if you had more time


**Write your final reflection here.**


## 📤 Final Deliverables Checklist

Before submitting your work, verify that you are delivering:

- [ ] completed notebook
- [ ] generated submission file
- [ ] leaderboard score recorded
- [ ] short discussion of preprocessing and model choices
- [ ] final reflection


## Report section


Lalala
